# Document-Grounded Q&A System for Long Documents

**Goal:** Answer questions accurately from a single long document (100+ pages) with citations and hallucination resistance.

**Pipeline:**
1. **Ingestion** — Parse PDF, extract text + metadata per page
2. **Chunking** — Split into overlapping, section-aware chunks
3. **Indexing** — FAISS (semantic) + BM25 (keyword) dual index
4. **Retrieval** — Hybrid search with Reciprocal Rank Fusion
5. **Generation** — LLM with strict grounding prompt + citations
6. **Evaluation** — Automated benchmark against FinanceBench gold answers + adversarial tests

### Evaluation Strategy
We evaluate on **8-10 documents** from [FinanceBench](https://github.com/patronus-ai/financebench),
which provides human-annotated gold answers. For each document, we:
- Run all matching FinanceBench questions and compare to gold answers
- Run adversarial "nonsense" questions that have no answer in the document (should trigger refusal)
- Score faithfulness, relevance, citation quality, and correctness
- Output a comprehensive `results.md` report

---
## 1. Setup

In [ ]:
!pip install pymupdf sentence-transformers faiss-cpu rank_bm25 openai tiktoken -q

In [40]:
import os
import re
import json
import hashlib
import time
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple, Any
from pathlib import Path
import warnings
from dotenv import load_dotenv
warnings.filterwarnings("ignore")

import fitz  # PyMuPDF
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from openai import OpenAI
import tiktoken

load_dotenv()
print("All imports successful.")

All imports successful.


In [41]:
# ── Configuration ─────────────────────────────────────────
CONFIG = {
    # Paths
    "pdf_dir": "financebench/pdfs",                        # Directory containing FinanceBench PDFs
    "benchmark_file": "financebench/data/financebench_open_source.jsonl",  # FinanceBench QA data
    "results_file": "financebench/results.md",             # Output results file

    # Chunking
    "chunk_size": 512,          # tokens per chunk
    "chunk_overlap": 64,        # overlap tokens between chunks

    # Embedding
    "embedding_model": "BAAI/bge-base-en-v1.5",

    # Retrieval
    "top_k": 15,                # candidates from each retriever
    "top_k_final": 6,           # passages sent to LLM
    "rrf_k": 60,                # RRF smoothing constant

    # Generation
    "llm_model": "gpt-4o-mini",
    "temperature": 0.0,
    "max_tokens": 1024,

    # Evaluation
    "max_documents": 10,        # Max PDFs to evaluate
}

# API key
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("⚠️  OPENAI_API_KEY not set. Run: os.environ['OPENAI_API_KEY'] = 'sk-...'")
else:
    print("✅ OpenAI API key loaded.")

print(f"Embedding: {CONFIG['embedding_model']}")
print(f"Chunks: {CONFIG['chunk_size']} tokens, {CONFIG['chunk_overlap']} overlap")
print(f"LLM: {CONFIG['llm_model']}")
print(f"Max documents: {CONFIG['max_documents']}")

✅ OpenAI API key loaded.
Embedding: BAAI/bge-base-en-v1.5
Chunks: 512 tokens, 64 overlap
LLM: gpt-4o-mini
Max documents: 10


---
## 2. Data Structures

In [43]:
@dataclass
class PageElement:
    """A block of text extracted from one PDF page."""
    text: str
    page_num: int               # 1-indexed
    element_type: str           # 'heading', 'paragraph', 'table'
    section_title: str = ""     # nearest heading above this element


@dataclass
class Chunk:
    """A chunk of text ready for indexing and retrieval."""
    chunk_id: str
    text: str
    page_numbers: List[int]
    section_title: str
    token_count: int
    chunk_index: int

    def citation(self) -> str:
        pages = sorted(set(self.page_numbers))
        if len(pages) == 1:
            p = f"p. {pages[0]}"
        elif len(pages) <= 3:
            p = f"pp. {', '.join(map(str, pages))}"
        else:
            p = f"pp. {pages[0]}-{pages[-1]}"
        sec = f' — "{self.section_title}"' if self.section_title else ""
        return f"[{p}{sec}]"


@dataclass
class EvalResult:
    """Result of evaluating one question against one document."""
    doc_name: str
    question: str
    system_answer: str
    gold_answer: Optional[str]      # None for nonsense questions
    question_type: str              # 'benchmark', 'nonsense'
    sources_used: List[str]         # citation strings
    faithfulness: Optional[int]     # 1-5
    relevance: Optional[int]        # 1-5
    citation_quality: Optional[int] # 1-5
    correctness: Optional[int]      # 1-5 (vs gold answer, only for benchmark)
    abstained: bool                 # Did the system correctly refuse?
    eval_reasoning: str
    latency_seconds: float

print("Data structures defined.")

Data structures defined.


---
## 3. PDF Ingestion

In [44]:
def get_median_font_size(page) -> float:
    """Compute the median font size on a page for heading detection."""
    sizes = []
    blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)["blocks"]
    for b in blocks:
        if b.get("type", 1) != 0:
            continue
        for line in b.get("lines", []):
            for span in line.get("spans", []):
                t = span.get("text", "").strip()
                if len(t) > 2:
                    sizes.extend([span["size"]] * len(t))
    return float(np.median(sizes)) if sizes else 12.0


def detect_heading(block: dict, median_size: float) -> Tuple[bool, int]:
    """Heuristic heading detection based on font size vs page median."""
    lines = block.get("lines", [])
    if not lines:
        return False, 0

    font_sizes = []
    is_bold = False
    for line in lines:
        for span in line.get("spans", []):
            font_sizes.append(span["size"])
            if "bold" in span.get("font", "").lower():
                is_bold = True

    if not font_sizes:
        return False, 0

    block_size = max(font_sizes)
    ratio = block_size / median_size if median_size > 0 else 1.0

    full_text = " ".join(
        span["text"] for line in lines for span in line.get("spans", [])
    ).strip()

    if len(full_text) > 200 or len(full_text) < 2:
        return False, 0

    if ratio >= 1.5 or (ratio >= 1.3 and is_bold):
        return True, 1
    elif ratio >= 1.2 or (is_bold and len(full_text) < 80):
        return True, 2
    return False, 0


def extract_block_text(block: dict) -> str:
    """Get all text from a block's spans."""
    text = ""
    for line in block.get("lines", []):
        line_text = "".join(span["text"] for span in line.get("spans", []))
        text += line_text + "\n"
    return text.strip()


def ingest_pdf(pdf_path: str) -> Tuple[List[PageElement], dict]:
    """Parse a PDF and return structured elements with metadata."""
    doc = fitz.open(pdf_path)
    elements = []
    current_section = ""

    metadata = {
        "title": doc.metadata.get("title", ""),
        "author": doc.metadata.get("author", ""),
        "total_pages": len(doc),
        "file_name": os.path.basename(pdf_path),
    }

    for page_idx in range(len(doc)):
        page = doc[page_idx]
        page_num = page_idx + 1
        median_fs = get_median_font_size(page)
        blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)["blocks"]

        try:
            tables = page.find_tables()
            for table in tables:
                df = table.to_pandas()
                if not df.empty:
                    elements.append(PageElement(
                        text=df.to_markdown(index=False),
                        page_num=page_num,
                        element_type="table",
                        section_title=current_section,
                    ))
        except Exception:
            pass

        for block in blocks:
            if block.get("type", 1) != 0:
                continue

            text = extract_block_text(block)
            if not text or len(text) < 3:
                continue

            is_heading, level = detect_heading(block, median_fs)
            if is_heading:
                current_section = text.strip()
                elements.append(PageElement(
                    text=text, page_num=page_num,
                    element_type="heading", section_title=current_section,
                ))
            else:
                elements.append(PageElement(
                    text=text, page_num=page_num,
                    element_type="paragraph", section_title=current_section,
                ))

    doc.close()
    return elements, metadata

print("Ingestion functions defined.")

Ingestion functions defined.


---
## 4. Chunking

In [45]:
tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text))


def chunk_elements(elements: List[PageElement], chunk_size=512, overlap=64) -> List[Chunk]:
    """Convert elements into overlapping, section-aware chunks."""
    chunks = []
    chunk_idx = 0

    # Group by section
    sections = []
    current_title = ""
    current_elems = []
    for elem in elements:
        if elem.element_type == "heading":
            if current_elems:
                sections.append((current_title, current_elems))
            current_title = elem.text.strip()
            current_elems = [elem]
        else:
            current_elems.append(elem)
    if current_elems:
        sections.append((current_title, current_elems))

    for section_title, section_elems in sections:
        buf_texts, buf_pages, buf_tokens = [], [], 0

        for elem in section_elems:
            elem_tokens = count_tokens(elem.text)

            if elem_tokens > chunk_size:
                if buf_texts:
                    chunks.append(_make_chunk(buf_texts, buf_pages, section_title, chunk_idx))
                    chunk_idx += 1
                    buf_texts, buf_pages, buf_tokens = [], [], 0
                tokens = tokenizer.encode(elem.text)
                for i in range(0, len(tokens), chunk_size):
                    piece = tokenizer.decode(tokens[i:i + chunk_size])
                    chunks.append(_make_chunk([piece], [elem.page_num], section_title, chunk_idx))
                    chunk_idx += 1
                continue

            if buf_tokens + elem_tokens > chunk_size and buf_texts:
                chunks.append(_make_chunk(buf_texts, buf_pages, section_title, chunk_idx))
                chunk_idx += 1
                keep_texts, keep_pages, keep_tokens = [], [], 0
                for t, p in zip(reversed(buf_texts), reversed(buf_pages)):
                    t_tok = count_tokens(t)
                    if keep_tokens + t_tok > overlap:
                        break
                    keep_texts.insert(0, t)
                    keep_pages.insert(0, p)
                    keep_tokens += t_tok
                buf_texts, buf_pages, buf_tokens = keep_texts, keep_pages, keep_tokens

            buf_texts.append(elem.text)
            buf_pages.append(elem.page_num)
            buf_tokens += elem_tokens

        if buf_texts:
            chunks.append(_make_chunk(buf_texts, buf_pages, section_title, chunk_idx))
            chunk_idx += 1

    chunks = [c for c in chunks if c.token_count >= 30]
    return chunks


def _make_chunk(texts, pages, section_title, idx):
    combined = "\n\n".join(texts)
    cid = hashlib.md5(f"{idx}:{combined[:50]}".encode()).hexdigest()[:10]
    return Chunk(
        chunk_id=cid, text=combined, page_numbers=list(pages),
        section_title=section_title, token_count=count_tokens(combined), chunk_index=idx,
    )

print("Chunking functions defined.")

Chunking functions defined.


---
## 5. Hybrid Index & Retrieval

Two complementary indices fused with Reciprocal Rank Fusion:
- **FAISS** (dense embeddings) — semantic similarity, paraphrases
- **BM25** (keyword matching) — exact terms, numbers, acronyms

In [46]:
class HybridIndex:
    """FAISS + BM25 hybrid index with RRF fusion."""

    def __init__(self, embed_model: SentenceTransformer):
        self.embed_model = embed_model
        self.chunks: List[Chunk] = []
        self.faiss_index = None
        self.bm25 = None

    def build(self, chunks: List[Chunk]):
        self.chunks = chunks
        texts = [c.text for c in chunks]

        # Dense index
        embeddings = self.embed_model.encode(
            texts, show_progress_bar=False, batch_size=32, normalize_embeddings=True
        )
        dim = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dim)
        self.faiss_index.add(embeddings.astype(np.float32))

        # Sparse index
        tokenized = [self._tokenize(t) for t in texts]
        self.bm25 = BM25Okapi(tokenized)

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
        return [t for t in text.split() if len(t) > 1]

    def search(self, query: str, top_k: int = 15, rrf_k: int = 60) -> List[Tuple[Chunk, float]]:
        """Hybrid search: FAISS + BM25 combined with RRF."""
        # Dense
        q_emb = self.embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
        scores_d, indices_d = self.faiss_index.search(q_emb, top_k)
        dense_ranking = [(int(idx), rank) for rank, idx in enumerate(indices_d[0]) if idx >= 0]

        # Sparse
        q_tokens = self._tokenize(query)
        bm25_scores = self.bm25.get_scores(q_tokens)
        top_sparse = np.argsort(bm25_scores)[::-1][:top_k]
        sparse_ranking = [(int(idx), rank) for rank, idx in enumerate(top_sparse) if bm25_scores[idx] > 0]

        # RRF fusion
        rrf = {}
        for idx, rank in dense_ranking:
            rrf[idx] = rrf.get(idx, 0) + 1.0 / (rrf_k + rank + 1)
        for idx, rank in sparse_ranking:
            rrf[idx] = rrf.get(idx, 0) + 1.0 / (rrf_k + rank + 1)

        sorted_results = sorted(rrf.items(), key=lambda x: x[1], reverse=True)
        return [(self.chunks[idx], score) for idx, score in sorted_results[:top_k]]

print("HybridIndex defined.")

HybridIndex defined.


---
## 6. Answer Generation

In [47]:
SYSTEM_PROMPT = """You are a precise document analysis assistant. Answer questions based
ONLY on the provided source passages.

RULES:
1. Use ONLY information from the provided sources. No external knowledge.
2. Cite sources using [Source N] notation.
3. If multiple sources support a claim, cite all: [Source 1, Source 3].
4. If the sources don't contain enough information to answer the question, say:
   "The provided document sections do not contain sufficient information to answer this question."
5. Do not speculate beyond what the sources explicitly state.
6. For numbers, quote exact figures from the sources.
7. Be concise but complete."""


def generate_answer(query: str, passages: List[Tuple[Chunk, float]], config: dict) -> dict:
    """Generate a grounded answer with citations."""
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

    context_parts = []
    for i, (chunk, score) in enumerate(passages, 1):
        context_parts.append(f"[Source {i}] {chunk.citation()}\n{chunk.text}")
    context = "\n\n---\n\n".join(context_parts)

    user_msg = f"""Based on the following source passages, answer the question.

SOURCE PASSAGES:
{context}

QUESTION: {query}

Provide a precise, well-cited answer using only the sources above."""

    response = client.chat.completions.create(
        model=config["llm_model"],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
    )

    return {
        "query": query,
        "answer": response.choices[0].message.content,
        "sources": [
            {"source_num": i + 1, "chunk": chunk, "score": score}
            for i, (chunk, score) in enumerate(passages)
        ],
        "usage": {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        },
    }


def ask(query: str, index: HybridIndex, config: dict) -> dict:
    """Full pipeline: retrieve → generate → cite."""
    candidates = index.search(query, top_k=config["top_k"], rrf_k=config["rrf_k"])
    passages = candidates[:config["top_k_final"]]
    return generate_answer(query, passages, config)

print("Generation functions defined.")

Generation functions defined.


---
## 7. FinanceBench Data Loader

Load the benchmark data and match questions to PDFs.
We also define adversarial "nonsense" questions — questions that have
absolutely nothing to do with any financial document. A robust system
should refuse to answer these rather than hallucinate.

In [48]:
# Adversarial nonsense questions — these have NO answer in any financial document.
# The system should abstain or explicitly say it cannot find relevant information.
NONSENSE_QUESTIONS = [
    "What is the meaning of life?",
    "Who invented algebra?",
    "What is the recipe for chocolate cake?",
    "How far is the Earth from the Sun?",
    "What is the plot of Romeo and Juliet?",
    "Who won the 2022 FIFA World Cup?",
    "What is the chemical formula for water?",
    "How do you train a golden retriever puppy?",
]

# Phrases that indicate the system correctly refused to answer
ABSTENTION_PHRASES = [
    "do not contain sufficient information",
    "does not contain sufficient information",
    "not contain enough information",
    "cannot find",
    "no information",
    "not mentioned",
    "not found in",
    "cannot be determined",
    "cannot be answered",
    "not available in",
    "not discussed in",
    "no relevant information",
    "outside the scope",
    "not addressed",
    "does not appear",
    "i cannot answer",
    "unable to answer",
    "not possible to answer",
    "i don't have",
]


def load_financebench(benchmark_file: str) -> List[dict]:
    """Load FinanceBench questions from JSONL file."""
    questions = []
    with open(benchmark_file, "r") as f:
        for line in f:
            if line.strip():
                questions.append(json.loads(line))
    print(f"✅ Loaded {len(questions)} FinanceBench questions")
    return questions


def get_questions_for_doc(all_questions: List[dict], doc_name: str) -> List[dict]:
    """Filter benchmark questions that match a specific document."""
    return [q for q in all_questions if q["doc_name"] == doc_name]


def find_pdf_for_doc(pdf_dir: str, doc_name: str) -> Optional[str]:
    """Find the PDF file matching a FinanceBench doc_name."""
    # FinanceBench PDFs are typically named like the doc_name with .pdf extension
    candidates = [
        os.path.join(pdf_dir, f"{doc_name}.pdf"),
        os.path.join(pdf_dir, f"{doc_name.lower()}.pdf"),
    ]
    # Also try matching by pattern
    for f in os.listdir(pdf_dir):
        if f.endswith(".pdf") and doc_name.lower() in f.lower():
            candidates.append(os.path.join(pdf_dir, f))

    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def detect_abstention(answer: str) -> bool:
    """Check if the system's answer is a refusal / abstention."""
    answer_lower = answer.lower()
    return any(phrase in answer_lower for phrase in ABSTENTION_PHRASES)


def select_documents(all_questions: List[dict], pdf_dir: str, max_docs: int = 10) -> List[dict]:
    """
    Select documents for evaluation, prioritizing:
    1. Documents with the most benchmark questions (more evaluation signal)
    2. Documents from different companies (diversity)
    3. Only 10-K filings (100+ pages, per the test requirements)
    """
    # Count questions per document
    doc_counts = {}
    for q in all_questions:
        doc_name = q["doc_name"]
        doc_counts[doc_name] = doc_counts.get(doc_name, 0) + 1

    # Filter to docs we actually have PDFs for, prefer 10K filings
    available = []
    for doc_name, count in sorted(doc_counts.items(), key=lambda x: x[1], reverse=True):
        pdf_path = find_pdf_for_doc(pdf_dir, doc_name)
        if pdf_path:
            is_10k = "10K" in doc_name or "10-K" in doc_name
            available.append({
                "doc_name": doc_name,
                "pdf_path": pdf_path,
                "num_questions": count,
                "is_10k": is_10k,
                "company": q.get("company", doc_name.split("_")[0]),
            })

    # Prefer 10-K filings, then by question count
    available.sort(key=lambda x: (x["is_10k"], x["num_questions"]), reverse=True)

    # Try to pick diverse companies
    selected = []
    seen_companies = set()
    for doc in available:
        if len(selected) >= max_docs:
            break
        company = doc["company"]
        if company not in seen_companies or len(selected) < max_docs:
            selected.append(doc)
            seen_companies.add(company)

    return selected[:max_docs]

print("FinanceBench loader defined.")

FinanceBench loader defined.


---
## 8. Evaluation Engine

Two types of evaluation:

1. **Benchmark questions** — Compare system answer to gold answer on:
   - Faithfulness (grounded in sources?)
   - Relevance (answers the question?)
   - Citation quality (proper citations?)
   - Correctness (matches gold answer?)

2. **Nonsense questions** — Check if the system correctly refuses:
   - Should detect that the question is unanswerable from the document
   - Should abstain rather than hallucinate

In [49]:
EVAL_BENCHMARK_PROMPT = """You are evaluating a document Q&A system. Compare the system's answer
to the gold (correct) answer, and rate on four dimensions (1-5 each).

QUESTION: {question}

GOLD ANSWER: {gold_answer}

SYSTEM ANSWER: {system_answer}

RETRIEVED SOURCES (snippets):
{sources}

Rate each dimension:
1. FAITHFULNESS (1-5): Is the system answer supported by the retrieved sources?
   5=fully grounded, 1=contains hallucinated claims
2. RELEVANCE (1-5): Does it address the question asked?
   5=directly answers, 1=off-topic
3. CITATION_QUALITY (1-5): Are [Source N] citations present and correct?
   5=all claims cited correctly, 1=no citations
4. CORRECTNESS (1-5): Does the system answer match the gold answer in substance?
   5=matches perfectly, 3=partially correct, 1=completely wrong

Respond ONLY with JSON:
{{"faithfulness": <1-5>, "relevance": <1-5>, "citation_quality": <1-5>, "correctness": <1-5>, "reasoning": "<brief explanation>"}}"""


EVAL_NONSENSE_PROMPT = """You are evaluating whether a document Q&A system correctly REFUSED to answer
an unanswerable question. The question below has NOTHING to do with the document.

QUESTION: {question}
SYSTEM ANSWER: {system_answer}

Did the system correctly refuse or abstain? Rate:
1. FAITHFULNESS (1-5): Did it avoid making up information? 5=correctly refused, 1=hallucinated an answer
2. RELEVANCE (1-5): Did it recognize the question is unanswerable from the document? 5=yes, 1=no

Respond ONLY with JSON:
{{"faithfulness": <1-5>, "relevance": <1-5>, "reasoning": "<brief explanation>"}}"""


def evaluate_benchmark_question(result: dict, gold_answer: str, config: dict) -> dict:
    """Evaluate a benchmark question against gold answer."""
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

    sources_text = ""
    for s in result.get("sources", []):
        c = s["chunk"]
        sources_text += f"[Source {s['source_num']}] {c.citation()}\n{c.text[:300]}\n\n"

    prompt = EVAL_BENCHMARK_PROMPT.format(
        question=result["query"],
        gold_answer=gold_answer,
        system_answer=result["answer"],
        sources=sources_text[:3000],  # cap to avoid token overflow
    )

    try:
        response = client.chat.completions.create(
            model=config["llm_model"],
            temperature=0.0,
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.choices[0].message.content
        match = re.search(r'\{[^}]+\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        return {"error": str(e)}
    return {"error": "Failed to parse"}


def evaluate_nonsense_question(result: dict, config: dict) -> dict:
    """Evaluate whether the system correctly refused a nonsense question."""
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

    prompt = EVAL_NONSENSE_PROMPT.format(
        question=result["query"],
        system_answer=result["answer"],
    )

    try:
        response = client.chat.completions.create(
            model=config["llm_model"],
            temperature=0.0,
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.choices[0].message.content
        match = re.search(r'\{[^}]+\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        return {"error": str(e)}
    return {"error": "Failed to parse"}

print("Evaluation functions defined.")

Evaluation functions defined.


---
## 9. Full Benchmark Runner

This is the main evaluation loop. For each selected document:
1. Ingest the PDF
2. Chunk and build the index
3. Run all matching FinanceBench questions
4. Run all nonsense questions
5. Evaluate each answer
6. Collect all results

In [51]:
def run_full_benchmark(config: dict) -> Tuple[List[EvalResult], dict]:
    """
    Run the full evaluation pipeline across multiple FinanceBench documents.

    Returns:
        eval_results: list of EvalResult for every question
        summary: aggregate statistics
    """
    # Load benchmark data
    print("Loading FinanceBench data...")
    all_questions = load_financebench(config["benchmark_file"])

    # Select documents
    print("\nSelecting documents...")
    selected_docs = select_documents(all_questions, config["pdf_dir"], config["max_documents"])
    print(f"Selected {len(selected_docs)} documents:")
    for d in selected_docs:
        print(f"  {d['doc_name']} ({d['num_questions']} questions, {'10-K' if d['is_10k'] else 'other'})")

    # Load embedding model once (shared across all documents)
    print(f"\nLoading embedding model: {config['embedding_model']}...")
    embed_model = SentenceTransformer(config["embedding_model"])
    print("✅ Model loaded.")

    all_eval_results = []
    doc_summaries = []

    for doc_idx, doc_info in enumerate(selected_docs):
        doc_name = doc_info["doc_name"]
        pdf_path = doc_info["pdf_path"]

        print(f"\n{'='*70}")
        print(f"[{doc_idx+1}/{len(selected_docs)}] Processing: {doc_name}")
        print(f"{'='*70}")

        # ── Ingest ────────────────────────────────────────────
        t0 = time.time()
        elements, metadata = ingest_pdf(pdf_path)
        print(f"  Ingested {metadata['total_pages']} pages → {len(elements)} elements")

        # ── Chunk ─────────────────────────────────────────────
        chunks = chunk_elements(elements, config["chunk_size"], config["chunk_overlap"])
        print(f"  Created {len(chunks)} chunks")

        # ── Index ─────────────────────────────────────────────
        index = HybridIndex(embed_model)
        index.build(chunks)
        ingest_time = time.time() - t0
        print(f"  Index built in {ingest_time:.1f}s")

        # ── Benchmark questions ───────────────────────────────
        doc_questions = get_questions_for_doc(all_questions, doc_name)
        print(f"\n  Running {len(doc_questions)} benchmark questions...")

        doc_results = {"benchmark": [], "nonsense": []}

        for q_idx, q_data in enumerate(doc_questions):
            question = q_data["question"]
            gold_answer = q_data["answer"]

            print(f"    [{q_idx+1}/{len(doc_questions)}] {question[:60]}...")

            t1 = time.time()
            result = ask(question, index, config)
            latency = time.time() - t1

            # Evaluate against gold answer
            eval_scores = evaluate_benchmark_question(result, gold_answer, config)
            abstained = detect_abstention(result["answer"])

            eval_result = EvalResult(
                doc_name=doc_name,
                question=question,
                system_answer=result["answer"],
                gold_answer=gold_answer,
                question_type="benchmark",
                sources_used=[s["chunk"].citation() for s in result["sources"]],
                faithfulness=eval_scores.get("faithfulness"),
                relevance=eval_scores.get("relevance"),
                citation_quality=eval_scores.get("citation_quality"),
                correctness=eval_scores.get("correctness"),
                abstained=abstained,
                eval_reasoning=eval_scores.get("reasoning", ""),
                latency_seconds=latency,
            )
            all_eval_results.append(eval_result)
            doc_results["benchmark"].append(eval_result)

        # ── Nonsense questions ────────────────────────────────
        # Pick 3 random nonsense questions per document
        nonsense_subset = NONSENSE_QUESTIONS[:3]
        print(f"\n  Running {len(nonsense_subset)} nonsense (adversarial) questions...")

        for q_idx, question in enumerate(nonsense_subset):
            print(f"    [{q_idx+1}/{len(nonsense_subset)}] {question[:60]}...")

            t1 = time.time()
            result = ask(question, index, config)
            latency = time.time() - t1

            abstained = detect_abstention(result["answer"])
            eval_scores = evaluate_nonsense_question(result, config)

            eval_result = EvalResult(
                doc_name=doc_name,
                question=question,
                system_answer=result["answer"],
                gold_answer=None,
                question_type="nonsense",
                sources_used=[s["chunk"].citation() for s in result["sources"]],
                faithfulness=eval_scores.get("faithfulness"),
                relevance=eval_scores.get("relevance"),
                citation_quality=None,
                correctness=None,
                abstained=abstained,
                eval_reasoning=eval_scores.get("reasoning", ""),
                latency_seconds=latency,
            )
            all_eval_results.append(eval_result)
            doc_results["nonsense"].append(eval_result)

        # ── Per-document summary ──────────────────────────────
        bench = doc_results["benchmark"]
        nonsense = doc_results["nonsense"]

        valid_bench = [r for r in bench if r.faithfulness is not None]
        doc_summary = {
            "doc_name": doc_name,
            "total_pages": metadata["total_pages"],
            "num_chunks": len(chunks),
            "ingest_time": ingest_time,
            "num_benchmark_qs": len(bench),
            "num_nonsense_qs": len(nonsense),
            "avg_faithfulness": np.mean([r.faithfulness for r in valid_bench]) if valid_bench else 0,
            "avg_relevance": np.mean([r.relevance for r in valid_bench]) if valid_bench else 0,
            "avg_citation": np.mean([r.citation_quality for r in valid_bench]) if valid_bench else 0,
            "avg_correctness": np.mean([r.correctness for r in valid_bench]) if valid_bench else 0,
            "nonsense_abstention_rate": sum(1 for r in nonsense if r.abstained) / len(nonsense) if nonsense else 0,
            "avg_latency": np.mean([r.latency_seconds for r in bench + nonsense]),
        }
        doc_summaries.append(doc_summary)

        print(f"\n  📊 {doc_name} Results:")
        print(f"     Faithfulness: {doc_summary['avg_faithfulness']:.1f}/5")
        print(f"     Relevance:    {doc_summary['avg_relevance']:.1f}/5")
        print(f"     Citations:    {doc_summary['avg_citation']:.1f}/5")
        print(f"     Correctness:  {doc_summary['avg_correctness']:.1f}/5")
        print(f"     Nonsense abstention: {doc_summary['nonsense_abstention_rate']:.0%}")

    # ── Global summary ────────────────────────────────────────
    summary = {
        "total_documents": len(selected_docs),
        "total_benchmark_qs": sum(d["num_benchmark_qs"] for d in doc_summaries),
        "total_nonsense_qs": sum(d["num_nonsense_qs"] for d in doc_summaries),
        "overall_faithfulness": np.mean([d["avg_faithfulness"] for d in doc_summaries]),
        "overall_relevance": np.mean([d["avg_relevance"] for d in doc_summaries]),
        "overall_citation": np.mean([d["avg_citation"] for d in doc_summaries]),
        "overall_correctness": np.mean([d["avg_correctness"] for d in doc_summaries]),
        "overall_abstention_rate": np.mean([d["nonsense_abstention_rate"] for d in doc_summaries]),
        "overall_avg_latency": np.mean([d["avg_latency"] for d in doc_summaries]),
        "doc_summaries": doc_summaries,
    }

    return all_eval_results, summary

print("Benchmark runner defined.")

Benchmark runner defined.


---
## 10. Results Report Generator

Generates a comprehensive `results.md` with:
- Overall scores
- Per-document breakdown
- Per-question details
- Nonsense question analysis
- Configuration used

In [52]:
def generate_results_md(
    eval_results: List[EvalResult],
    summary: dict,
    config: dict,
    output_path: str = "./results.md"
):
    """Generate a comprehensive results.md report."""

    lines = []

    def add(text=""):
        lines.append(text)

    # ── Header ────────────────────────────────────────────────
    add("# Document Q&A System — Evaluation Results")
    add()
    add(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    add(f"**Benchmark:** FinanceBench (open-source subset)")
    add(f"**Documents evaluated:** {summary['total_documents']}")
    add(f"**Benchmark questions:** {summary['total_benchmark_qs']}")
    add(f"**Adversarial (nonsense) questions:** {summary['total_nonsense_qs']}")
    add()

    # ── Overall Scores ────────────────────────────────────────
    add("---")
    add("## Overall Scores")
    add()
    add("| Metric | Score |")
    add("|--------|-------|")
    add(f"| Faithfulness | **{summary['overall_faithfulness']:.2f}** / 5 |")
    add(f"| Relevance | **{summary['overall_relevance']:.2f}** / 5 |")
    add(f"| Citation Quality | **{summary['overall_citation']:.2f}** / 5 |")
    add(f"| Correctness (vs gold) | **{summary['overall_correctness']:.2f}** / 5 |")
    add(f"| Nonsense Abstention Rate | **{summary['overall_abstention_rate']:.0%}** |")
    add(f"| Average Latency | **{summary['overall_avg_latency']:.1f}s** per question |")
    add()

    # ── Scoring Guide ─────────────────────────────────────────
    add("> **Scoring guide:** Each dimension is rated 1-5 by an LLM judge.")
    add("> Faithfulness = grounded in sources. Relevance = answers the question.")
    add("> Citation Quality = proper [Source N] references. Correctness = matches gold answer.")
    add("> Abstention Rate = % of nonsense questions correctly refused (higher is better).")
    add()

    # ── Per-Document Breakdown ────────────────────────────────
    add("---")
    add("## Per-Document Results")
    add()
    add("| Document | Pages | Chunks | Qs | Faith. | Relev. | Cite. | Correct. | Abstain | Latency |")
    add("|----------|-------|--------|----|--------|--------|-------|----------|---------|---------|")

    for d in summary["doc_summaries"]:
        add(f"| {d['doc_name']} | {d['total_pages']} | {d['num_chunks']} | "
            f"{d['num_benchmark_qs']} | {d['avg_faithfulness']:.1f} | {d['avg_relevance']:.1f} | "
            f"{d['avg_citation']:.1f} | {d['avg_correctness']:.1f} | "
            f"{d['nonsense_abstention_rate']:.0%} | {d['avg_latency']:.1f}s |")
    add()

    # ── Detailed Results per Document ─────────────────────────
    add("---")
    add("## Detailed Results")
    add()

    # Group results by document
    docs = {}
    for r in eval_results:
        if r.doc_name not in docs:
            docs[r.doc_name] = {"benchmark": [], "nonsense": []}
        docs[r.doc_name][r.question_type].append(r)

    for doc_name, results_by_type in docs.items():
        add(f"### {doc_name}")
        add()

        # Benchmark questions
        if results_by_type["benchmark"]:
            add("#### Benchmark Questions")
            add()
            for i, r in enumerate(results_by_type["benchmark"], 1):
                status = "✅" if (r.correctness and r.correctness >= 4) else "⚠️" if (r.correctness and r.correctness >= 3) else "❌"
                add(f"**{status} Q{i}: {r.question}**")
                add()
                add(f"- **Gold answer:** {r.gold_answer[:200]}{'...' if len(r.gold_answer or '') > 200 else ''}")
                add(f"- **System answer:** {r.system_answer[:200]}{'...' if len(r.system_answer) > 200 else ''}")
                add(f"- **Scores:** Faith={r.faithfulness} Rel={r.relevance} Cite={r.citation_quality} Correct={r.correctness}")
                add(f"- **Sources:** {', '.join(r.sources_used[:3])}")
                add(f"- **Judge reasoning:** {r.eval_reasoning[:150]}")
                add(f"- **Latency:** {r.latency_seconds:.1f}s")
                add()

        # Nonsense questions
        if results_by_type["nonsense"]:
            add("#### Adversarial (Nonsense) Questions")
            add()
            for i, r in enumerate(results_by_type["nonsense"], 1):
                status = "✅" if r.abstained else "❌"
                add(f"**{status} Q{i}: {r.question}**")
                add()
                add(f"- **Abstained:** {'Yes ✅' if r.abstained else 'No ❌ (should have refused)'}")
                add(f"- **System answer:** {r.system_answer[:200]}{'...' if len(r.system_answer) > 200 else ''}")
                add(f"- **Scores:** Faith={r.faithfulness} Rel={r.relevance}")
                add(f"- **Judge reasoning:** {r.eval_reasoning[:150]}")
                add()

    # ── Configuration ─────────────────────────────────────────
    add("---")
    add("## Configuration")
    add()
    add("```json")
    config_display = {k: v for k, v in config.items() if k != "openai_api_key"}
    add(json.dumps(config_display, indent=2))
    add("```")
    add()

    # ── Architecture ──────────────────────────────────────────
    add("---")
    add("## System Architecture")
    add()
    add("```")
    add("PDF → PyMuPDF Parser → Section-Aware Chunker → [FAISS + BM25] → RRF Fusion → LLM + Citations")
    add("```")
    add()
    add("| Component | Choice |")
    add("|-----------|--------|")
    add(f"| Embedding | {config['embedding_model']} |")
    add(f"| Chunk size | {config['chunk_size']} tokens (overlap {config['chunk_overlap']}) |")
    add(f"| Retrieval | Hybrid FAISS + BM25 with RRF (k={config['rrf_k']}) |")
    add(f"| Top-k | {config['top_k']} candidates → {config['top_k_final']} to LLM |")
    add(f"| LLM | {config['llm_model']} (temp={config['temperature']}) |")
    add()

    # Write file
    report = "\n".join(lines)
    with open(output_path, "w") as f:
        f.write(report)

    print(f"\n✅ Results written to: {output_path}")
    print(f"   ({len(eval_results)} questions across {summary['total_documents']} documents)")

    return report

print("Results generator defined.")

Results generator defined.


---
## 11. Run the Full Benchmark

**Before running:**
1. Clone FinanceBench: `git clone https://github.com/patronus-ai/financebench.git`
2. Set paths below to point to the cloned repo
3. Set your OpenAI API key

This will process 8-10 documents automatically, run all matching questions + nonsense questions,
and generate `results.md`.

In [53]:
# ── Configure paths ───────────────────────────────────────
# Adjust these to point to your FinanceBench clone:

CONFIG["pdf_dir"] = "./financebench/pdfs"                               # ← PDF directory
CONFIG["benchmark_file"] = "./financebench/data/financebench_open_source.jsonl"  # ← QA data
CONFIG["results_file"] = "./results.md"                                 # ← Output file
CONFIG["max_documents"] = 10                                            # ← Number of docs

# Verify paths
pdf_dir_ok = os.path.isdir(CONFIG["pdf_dir"])
bench_ok = os.path.isfile(CONFIG["benchmark_file"])

print(f"PDF directory:    {'✅' if pdf_dir_ok else '❌'} {CONFIG['pdf_dir']}")
print(f"Benchmark file:   {'✅' if bench_ok else '❌'} {CONFIG['benchmark_file']}")
print(f"API key:          {'✅' if OPENAI_API_KEY else '❌'}")

if pdf_dir_ok:
    pdfs = [f for f in os.listdir(CONFIG["pdf_dir"]) if f.endswith(".pdf")]
    print(f"PDFs found:       {len(pdfs)}")

if not all([pdf_dir_ok, bench_ok, OPENAI_API_KEY]):
    print("\n⚠️  Fix the issues above before running the benchmark.")

PDF directory:    ✅ ./financebench/pdfs
Benchmark file:   ✅ ./financebench/data/financebench_open_source.jsonl
API key:          ✅
PDFs found:       13


In [54]:
# ── RUN THE BENCHMARK ─────────────────────────────────────
# This will take a while (~2-5 min per document depending on size).

if all([os.path.isdir(CONFIG["pdf_dir"]),
        os.path.isfile(CONFIG["benchmark_file"]),
        OPENAI_API_KEY]):

    print("🚀 Starting full benchmark run...")
    print(f"   This will process up to {CONFIG['max_documents']} documents.\n")

    start_time = time.time()
    eval_results, summary = run_full_benchmark(CONFIG)
    total_time = time.time() - start_time

    # Print summary
    print(f"\n{'='*70}")
    print(f"BENCHMARK COMPLETE")
    print(f"{'='*70}")
    print(f"Documents:  {summary['total_documents']}")
    print(f"Questions:  {summary['total_benchmark_qs']} benchmark + {summary['total_nonsense_qs']} nonsense")
    print(f"Total time: {total_time/60:.1f} minutes")
    print()
    print(f"📊 OVERALL SCORES:")
    print(f"   Faithfulness:     {summary['overall_faithfulness']:.2f}/5")
    print(f"   Relevance:        {summary['overall_relevance']:.2f}/5")
    print(f"   Citation Quality: {summary['overall_citation']:.2f}/5")
    print(f"   Correctness:      {summary['overall_correctness']:.2f}/5")
    print(f"   Abstention Rate:  {summary['overall_abstention_rate']:.0%}")

    # Generate results.md
    report = generate_results_md(eval_results, summary, CONFIG, CONFIG["results_file"])
else:
    print("⚠️  Cannot run benchmark. Check paths and API key above.")

🚀 Starting full benchmark run...
   This will process up to 10 documents.

Loading FinanceBench data...
✅ Loaded 150 FinanceBench questions

Selecting documents...
Selected 4 documents:
  AMD_2022_10K (7 questions, 10-K)
  ADOBE_2017_10K (1 questions, 10-K)
  AMCOR_2020_10K (1 questions, 10-K)
  COCACOLA_2022_10K (1 questions, 10-K)

Loading embedding model: BAAI/bge-base-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15171.07it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded.

[1/4] Processing: AMD_2022_10K
  Ingested 121 pages → 1689 elements
  Created 359 chunks
  Index built in 15.2s

  Running 7 benchmark questions...
    [1/7] Does AMD have a reasonably healthy liquidity profile based o...
    [2/7] What are the major products and services that AMD sells as o...
    [3/7] What drove revenue change as of the FY22 for AMD?...
    [4/7] What drove operating margin change as of the FY22 for AMD? I...
    [5/7] Among operations, investing, and financing activities, which...
    [6/7] From FY21 to FY22, excluding Embedded, in which AMD reportin...
    [7/7] Did AMD report customer concentration in FY22?...

  Running 3 nonsense (adversarial) questions...
    [1/3] What is the meaning of life?...
    [2/3] Who invented algebra?...
    [3/3] What is the recipe for chocolate cake?...

  📊 AMD_2022_10K Results:
     Faithfulness: 4.0/5
     Relevance:    3.9/5
     Citations:    3.9/5
     Correctness:  3.6/5
     Nonsense abstention: 100%

[2/4]

---
## 12. Single Document Mode (Manual)

Use this section if you want to query a single document interactively,
outside of the benchmark.

In [ ]:
# ── Single Document Q&A ───────────────────────────────────
# For manual testing on any PDF.

SINGLE_PDF = "document.pdf"  # ← Change to any PDF path

if os.path.exists(SINGLE_PDF):
    print(f"Processing: {SINGLE_PDF}")
    elements, metadata = ingest_pdf(SINGLE_PDF)
    print(f"  {metadata['total_pages']} pages, {len(elements)} elements")

    chunks = chunk_elements(elements, CONFIG["chunk_size"], CONFIG["chunk_overlap"])
    print(f"  {len(chunks)} chunks")

    embed_model = SentenceTransformer(CONFIG["embedding_model"])
    single_index = HybridIndex(embed_model)
    single_index.build(chunks)

    # Ask a question
    result = ask("What was the total revenue?", single_index, CONFIG)
    print(f"\n💡 Answer: {result['answer'][:300]}")
    print(f"\n📖 Sources:")
    for s in result["sources"]:
        print(f"   [Source {s['source_num']}] {s['chunk'].citation()}")
else:
    print(f"⚠️  File not found: {SINGLE_PDF}")
    print("   Set SINGLE_PDF to test with a specific document.")

In [ ]:
# ── Interactive Q&A ───────────────────────────────────────
def interactive_qa(index):
    print("╔════════════════════════════════════════════╗")
    print("║   Document Q&A — Interactive Mode          ║")
    print("║   Type 'quit' to exit                      ║")
    print("╚════════════════════════════════════════════╝")
    while True:
        print()
        q = input("❓ Question: ").strip()
        if q.lower() in ("quit", "exit", "q"):
            print("👋 Done!")
            break
        if not q:
            continue
        result = ask(q, index, CONFIG)
        print(f"\n💡 {result['answer']}")
        print(f"\n📖 Sources:")
        for s in result["sources"]:
            print(f"   [Source {s['source_num']}] {s['chunk'].citation()}")

# Uncomment to start interactive mode (requires single_index from above):
# interactive_qa(single_index)

---
## 13. Design Notes

### Architecture
```
PDF → PyMuPDF Parser → Section-Aware Chunker → [FAISS + BM25] → RRF Fusion → LLM + Citations
```

### Evaluation Design

| Test Type | Purpose | Success Criteria |
|-----------|---------|-----------------|
| Benchmark (FinanceBench) | Accuracy on real financial questions | High correctness vs gold answers |
| Adversarial (Nonsense) | Hallucination resistance | System refuses to answer |

**Why both?** A system that always answers sounds helpful but is dangerous — it will hallucinate when the answer isn't in the document. Testing with unanswerable questions reveals this failure mode.

### Key Design Decisions

| Decision | Choice | Why |
|----------|--------|-----|
| PDF Parser | PyMuPDF | Fast, structure-aware, table support |
| Chunking | Heading-grouped + token overlap | Preserves topical coherence |
| Dense Index | FAISS flat + bge-base-en-v1.5 | Strong retrieval, manageable size |
| Sparse Index | BM25 | Essential for exact terms, numbers |
| Fusion | RRF (k=60) | Score-agnostic, no normalization needed |
| LLM | gpt-4o-mini | Cost-effective, strong instruction following |
| Benchmark | FinanceBench | Gold answers, multiple reasoning types |
| Adversarial | Nonsense questions | Tests refusal/abstention capability |

### Limitations
1. Tables — Simple tables work; complex merged cells may fail
2. Images/Charts — Text-only extraction
3. Heading detection — Font-size heuristic, not perfect for all layouts
4. Evaluation — LLM-as-judge has known biases
5. Single document per query — by design

### Future Improvements
1. Cross-encoder re-ranker for better precision
2. Index caching to avoid re-embedding
3. Iterative retrieval for multi-hop questions
4. Semantic chunking using embedding similarity
5. Multimodal support for charts/images
6. Query decomposition for complex questions